# Pronóstico de caudal hídrico con LSTM multivariado

- Pronóstico de caudal a partir de datos hidrometeorológicos abiertos y reales: caudal observado y precipitación en tres estaciones, registros horarios de más de cuatro años.
- Se comparan una LSTM univariada autoregresiva, una LSTM multivariada, una variante con rezago específico por variable, y una extensión multi-step con arquitectura Encoder-Decoder.
- Artículo completo, con el diagrama del pipeline y las decisiones de ingeniería: https://fuzzyfrog.ai/es/ai-lab/proyectos/ambiente/pronostico-caudal-hidrico-lstm-multivariado/
- **Nota:** este notebook usa datos hidrometeorológicos abiertos y públicos. No contiene rutas de trabajo personales ni ningún dato que identifique al autor original del proyecto.


## Diagrama del pipeline

- Datos abiertos (caudal + lluvia, horario) → ventanas de rezago (uniformes y luego específicas por variable) → comparación de arquitecturas LSTM (univariada, multivariada, multi-step) → evaluación con métricas hidrológicas → pronóstico.
- Diagrama editable disponible en el artículo de la plataforma (liga arriba).


## Carga de datos

- Dataset: registros horarios de caudal en San Mateo y precipitación en tres estaciones (Casa, San Cristóbal, Ago), de abril 2018 a agosto 2022 (~38.450 registros).
- Datos hidrometeorológicos abiertos y públicos, incluidos en este repositorio (`SubCuenca_SanMateo.csv`).


In [ ]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler

datos = pd.read_csv("SubCuenca_SanMateo.csv", sep=",")
datos["Fecha_m"] = pd.to_datetime(datos["Fecha_m"], dayfirst=True)
datos = datos.dropna()
datos = datos.set_index("Fecha_m")

# Orden de columnas: variable objetivo primero, luego covariables
datos = datos.reindex(columns=["SanMat_q", "Casa_pp", "SanCri_pp", "Ago_pp"])
datos.head()


## Explicación de datos

- `SanMat_q`: caudal observado en San Mateo (m³/s), la variable a pronosticar.
- `Casa_pp`, `SanCri_pp`, `Ago_pp`: precipitación horaria (mm) en tres estaciones de la subcuenca.
- Tras eliminar filas con datos faltantes, quedan suficientes registros para dividir en entrenamiento, validación y prueba de forma secuencial, respetando el orden temporal.


In [ ]:
print(f"Registros totales tras limpieza: {len(datos)}")
print(datos.describe())


## Análisis de datos / EDA

- Antes de modelar, conviene ver la autocorrelación del caudal, para tener una primera idea de cuántas horas de historia aportan información.


In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plot_pacf(datos["SanMat_q"], lags=72)
plt.title("Autocorrelación parcial del caudal (hasta 72 horas)")
plt.show()


## Modelado

- **Línea base univariada:** LSTM autoregresiva, usando solo el caudal histórico con hasta 72 horas de rezago.
- **LSTM multivariada:** ventana uniforme de 8 horas para caudal y las 3 series de lluvia.
- **Variante con rezago por variable:** 8 horas para el caudal, 7 horas para cada estación de lluvia, reflejando que cada variable tiene su propia persistencia temporal.
- **Extensión multi-step:** arquitectura Encoder-Decoder para pronosticar varias horas hacia adelante, no solo la siguiente.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, LSTM, Dense, RepeatVector, TimeDistributed
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam

def df_to_X_y(df, window_size=8):
    df_as_np = df.to_numpy()
    X, y = [], []
    for i in range(len(df_as_np) - window_size):
        X.append([r for r in df_as_np[i:i + window_size]])
        y.append(df_as_np[i + window_size][0])  # SanMat_q es la primera columna
    return np.array(X), np.array(y)

# LSTM multivariada, ventana uniforme de 8 horas
X, y = df_to_X_y(datos, window_size=8)
X_train, y_train = X[:18000], y[:18000]
X_val, y_val = X[18000:24000], y[18000:24000]
X_test, y_test = X[24000:], y[24000:]

modelo_multivariado = Sequential([
    InputLayer((8, X.shape[2])),
    LSTM(64, return_sequences=True),
    LSTM(32, return_sequences=True),
    LSTM(64, return_sequences=False),
    Dense(8, activation="relu"),
    Dense(1, activation="linear"),
])
modelo_multivariado.compile(
    loss=MeanSquaredError(),
    optimizer=Adam(learning_rate=0.0001),
    metrics=[RootMeanSquaredError()],
)
modelo_multivariado.summary()


In [ ]:
# Entrenamiento (referencia; puede tardar varios minutos según el hardware disponible)
historial = modelo_multivariado.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
)


## Evaluación

- Se usan métricas estándar de hidrología, no solo error cuadrático medio genérico: Nash-Sutcliffe (NSE), BIAS, RMSE, r y r².
- Resultados reales del proyecto original, para comparar contra lo que obtengas al re-entrenar:

| Modelo | Nash train | Nash valid | Nash prueba |
|---|---|---|---|
| LSTM univariada (72h, solo caudal) | 0.981 | 0.978 | 0.716 |
| LSTM multivariada (8h, caudal + lluvia) | 0.986 | 0.982 | 0.758 |
| LSTM multivariada, rezago por variable | 0.981 | 0.978 | — (evaluado por ventana) |


In [ ]:
def metricas_hidrologicas(obs, sim):
    obs = np.array(obs)
    sim = np.array(sim)

    bias = round(float(np.sum(obs - sim) / np.sum(sim)), 4)
    nash = round(1 - np.sum((obs - sim) ** 2) / np.sum((obs - np.mean(obs)) ** 2), 3)
    rmse = round(math.sqrt(np.sum((obs - sim) ** 2) / len(obs)), 3)
    r = np.corrcoef(sim, obs)[0, 1]
    r2 = round(r ** 2, 3)

    return {"Nash": nash, "BIAS": bias, "RMSE": rmse, "r": round(r, 4), "r2": r2}

pred_test = modelo_multivariado.predict(X_test).flatten()
print("Métricas en prueba:", metricas_hidrologicas(y_test, pred_test))


In [ ]:
def graficar_dispersión(obs, sim, titulo):
    metrics = metricas_hidrologicas(obs, sim)
    fig = plt.figure(figsize=(5, 4), dpi=120)
    ax = fig.add_axes([0.1, 0.1, 0.8, 0.8])
    plt.plot(obs, sim, "o", color="black", mfc="white", mec="k", markersize=4)
    lims = [min(min(obs), min(sim)), max(max(obs), max(sim))]
    plt.plot(lims, lims, "r")
    plt.title(titulo, fontsize=10, color="darkblue", weight="bold")
    plt.xlabel("Q observados (m³/s)")
    plt.ylabel("Q simulados (m³/s)")
    plt.text(0.05, 0.7, "\n".join(f"{k}={v}" for k, v in metrics.items()),
              transform=ax.transAxes, fontsize=9, weight="bold")
    plt.grid(True)
    plt.show()

graficar_dispersión(y_test, pred_test, "DATOS DE PRUEBA")


## Hallazgos principales

- La LSTM multivariada (8 horas de ventana, con lluvia) alcanzó un Nash de prueba de 0.758, superando levemente a la línea base univariada autoregresiva (Nash de prueba 0.716, con 72 horas de rezago). La ganancia de agregar covariables de lluvia fue real, pero modesta.
- El desempeño cayó de forma notoria entre validación (Nash ≈ 0.98) y prueba (Nash ≈ 0.72–0.76) en ambos modelos. Esto sugiere que el periodo de prueba contiene un comportamiento hidrológico distinto al visto en entrenamiento, y se reporta así, con honestidad, en vez de solo mostrar las cifras más favorables.
- Ajustar el rezago por variable, en vez de usar una ventana uniforme, refleja mejor la física del problema, el caudal y la lluvia no tienen la misma persistencia temporal.
- El pronóstico a varias horas con la arquitectura Encoder-Decoder mostró el patrón esperado, el error crece con el horizonte de pronóstico, algo que cualquier proyecto de pronóstico multi-step debe anticipar y comunicar.
- Los datos hidrometeorológicos abiertos de gobierno son una fuente real y suficiente para un proyecto aplicado completo, sin necesidad de un dataset privado.
